# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Lane check (lanes lock this week):** I am staying in **Lane 2, Refresh / Content Opportunity Scoring**. Nothing in ML-04 or this notebook made me want to switch.

**The rule in plain words:** *a page is worth an editor's review first if it was already losing daily impressions inside March, and the more impressions per day it lost, the higher it goes. If it also gets fewer clicks than is normal for its position, it goes a bit higher still.*

**Data slice (same as ML-04, on purpose):** decision moment **31 March 2026**. Inputs come only from the March 2026 daily partition. Pages need GSC data on all 31 March days and all 30 April days, and at least 100 March impressions. The label is `is_declining` = 1 when April's average daily impressions are below 0.8 × March's average daily impressions.

**April is used only to grade, never as an input.** The April numbers (`imp_apr`, `is_declining`, impressions lost) appear only in the bucket tables below and in the precision@K evaluation in section 2. The score itself is computed from March columns only (checked in section 4).

**Why these two signals:**
1. **CTR vs position (flag-linked).** This is the signal behind FlyRank's CTR-fix logic: a page that gets fewer clicks than is normal for where it ranks. My idea: under-performing CTR is a warning sign, so those pages should decline more often in April.
2. **March momentum.** Late-March daily impressions (17-31 March) divided by early-March daily impressions (1-16 March). My idea: a page that was already sliding at the end of March keeps sliding in April.

**Why not staleness:** staleness sits behind the real refresh flags, but I cannot measure it honestly at 31 March. `content_updated_date` and `last_optimized_date` come from a snapshot taken after the panel, and in ML-04 about 78% of the pages in this slice showed a "last updated" date *after* 31 March. Any "days since update" built from them would quietly use the future, so they are not inputs here.

The setup cell below rebuilds the exact ML-04 feature frame.

In [1]:
import os, sys, subprocess, getpass, json

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "-q", "https://github.com/sohamrathi17/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    # Local run: walk up until I am at the repo root (the folder that holds data/raw).
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
print("Working from the repo root:", os.path.isdir("data/raw"))

# Token order: env var -> Colab Secret -> local `hf auth login` cache -> prompt (last resort).
# Never type the token into a cell: this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    from huggingface_hub import get_token
    HF_TOKEN = get_token()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download

# Each file is downloaded once into the Hugging Face cache (outside the repo, never committed).
def hf_file(path):
    return hf_hub_download("FlyRank/internship-warehouse", path, repo_type="dataset", token=HF_TOKEN)

MAR = f"read_parquet('{hf_file('fact_content_daily_performance/month=2026-03/data_0.parquet')}')"
APR = f"read_parquet('{hf_file('fact_content_daily_performance/month=2026-04/data_0.parquet')}')"
DIM_CONTENT = f"read_parquet('{hf_file('dim_content.parquet')}')"
con = duckdb.connect()

# Same SQL as ML-04: one row per page, March features, April only for the label.
df = con.sql(f"""
    WITH mar AS (
        SELECT client_hash_id, content_hash_id,
               COUNT(*)              AS gsc_days_mar,
               SUM(gsc_impressions)  AS imp_mar,
               SUM(gsc_clicks)       AS clk_mar,
               SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_sum_position END)::DOUBLE
                 / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END), 0) AS avg_position_mar,
               SUM(CASE WHEN report_date >= DATE '2026-03-17' THEN gsc_impressions END) / 15.0 AS imp_late_per_day,
               SUM(CASE WHEN report_date <  DATE '2026-03-17' THEN gsc_impressions END) / 16.0 AS imp_early_per_day
        FROM {MAR}
        WHERE gsc_data_available IS TRUE
        GROUP BY ALL
        HAVING COUNT(*) = 31 AND SUM(gsc_impressions) >= 100
    ),
    apr AS (   -- label window only: never used as a rule input
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
        FROM {APR}
        WHERE gsc_data_available IS TRUE
        GROUP BY ALL
        HAVING COUNT(*) = 30
    )
    SELECT m.*, a.imp_apr, d.content_created_date
    FROM mar m
    JOIN apr a USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
""").df()

DECISION = pd.Timestamp("2026-03-31")
df["log_imp_mar"]  = np.log1p(df["imp_mar"])
df["ctr_mar"]      = df["clk_mar"] / df["imp_mar"]          # a fraction, not x100
df["age_days"]     = (DECISION - pd.to_datetime(df["content_created_date"])).dt.days
df["momentum_mar"] = (df["imp_late_per_day"] + 1) / (df["imp_early_per_day"] + 1)

# Label + impact: used ONLY to grade signals and the rule.
df["is_declining"] = ((df["imp_apr"] / 30) < 0.8 * (df["imp_mar"] / 31)).astype(int)
df["imp_lost_apr"] = (df["imp_mar"] / 31 * 30 - df["imp_apr"]).clip(lower=0)

df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)   # fixed row order
assert df["content_hash_id"].is_unique, "grain broke: more than one row per page"
print(f"Pages: {len(df):,} | clients: {df['client_hash_id'].nunique()} | declining in April: {df['is_declining'].mean():.1%}")

Working from the repo root: True


Pages: 50,299 | clients: 32 | declining in April: 47.4%


### Signal check 1: CTR vs position (the CTR-fix signal)

CTR naturally falls as position gets worse, so raw CTR would mostly measure position. Instead I compare each page's March CTR to the **median March CTR of its position tier** (top_3 ≤ 3, page_1 ≤ 10, striking ≤ 20, page_3_5 ≤ 50, deep > 50). Buckets: zero clicks in March (kept apart, since a ratio means little there), under half the tier median, under the median, 1-2× the median, and 2× or more.

In [2]:
TIER_EDGES  = [0, 3, 10, 20, 50, np.inf]
TIER_LABELS = ["top_3", "page_1", "striking", "page_3_5", "deep"]
df["position_tier"] = pd.cut(df["avg_position_mar"], TIER_EDGES, labels=TIER_LABELS)

tier_median_ctr = df.groupby("position_tier", observed=True)["ctr_mar"].median()
df["ctr_vs_tier"] = df["ctr_mar"] / df["position_tier"].map(tier_median_ctr).astype(float)
df["ctr_bucket"] = np.select(
    [df["clk_mar"] == 0, df["ctr_vs_tier"] < 0.5, df["ctr_vs_tier"] < 1, df["ctr_vs_tier"] < 2],
    ["0_zero_clicks", "1_under_half", "2_under", "3_over_1_to_2x"], "4_over_2x_plus")

def bucket_table(col):
    t = df.groupby(col, observed=True).agg(
        n=("is_declining", "size"),
        decline_rate_apr=("is_declining", "mean"),
        median_imp_mar=("imp_mar", "median"),
        median_imp_lost_apr=("imp_lost_apr", "median"))
    return t.round({"decline_rate_apr": 3, "median_imp_mar": 0, "median_imp_lost_apr": 0})

print("Median March CTR per position tier:", (tier_median_ctr * 100).round(3).to_dict(), "(in %)")
print(f"n = {len(df):,} pages, overall April decline rate {df['is_declining'].mean():.3f}\n")
bucket_table("ctr_bucket")

Median March CTR per position tier: {'top_3': 0.25, 'page_1': 0.215, 'striking': 0.159, 'page_3_5': 0.067, 'deep': 0.0} (in %)
n = 50,299 pages, overall April decline rate 0.474



,n,decline_rate_apr,median_imp_mar,median_imp_lost_apr
ctr_bucket,,,,
0_zero_clicks,10363,0.492,610.0,94.0
1_under_half,6168,0.629,3522.0,850.0
2_under,8884,0.553,2640.0,447.0
3_over_1_to_2x,11206,0.474,2339.0,275.0
4_over_2x_plus,13678,0.340,2340.0,40.0


In [3]:
# Same buckets split by position tier, to check the pattern is not just one tier (n in the second table).
by_tier = df.groupby(["position_tier", "ctr_bucket"], observed=True)["is_declining"].agg(["mean", "size"])
print("April decline rate by tier x CTR bucket:")
print(by_tier["mean"].unstack().round(3))
print("\nn per cell:")
print(by_tier["size"].unstack().fillna(0).astype(int))

April decline rate by tier x CTR bucket:
ctr_bucket     0_zero_clicks  1_under_half  2_under  3_over_1_to_2x  \
position_tier                                                         
top_3                  0.627         0.757    0.687           0.538   
page_1                 0.524         0.616    0.532           0.442   
striking               0.493         0.617    0.535           0.493   
page_3_5               0.463         0.534    0.542           0.530   
deep                   0.372           NaN      NaN             NaN   

ctr_bucket     4_over_2x_plus  
position_tier                  
top_3                   0.280  
page_1                  0.267  
striking                0.377  
page_3_5                0.468  
deep                    0.444  

n per cell:
ctr_bucket     0_zero_clicks  1_under_half  2_under  3_over_1_to_2x  \
position_tier                                                         
top_3                    416           931     1135            1320   
page_1     

**Verdict: CONFIRMED** (for pages that have clicks).

The idea predicted that pages getting fewer clicks than normal for their position would decline more often. Among pages with at least one March click, the April decline rate falls steadily from about 0.63 (under half the tier median, n = 6,168) to 0.55 (under, n = 8,884), 0.47 (1-2×, n = 11,206) and 0.34 (2× or more, n = 13,678). The same order shows up inside each of the top_3, page_1 and striking tiers, so it is not just position in disguise. The one exception is the zero-click bucket (n = 10,363, rate about 0.49, close to the overall 0.47): these are mostly small pages, so "low CTR" there does not carry the same warning, and I do not give them the CTR boost. In page_3_5 the pattern is weaker, and the deep tier is tiny with a median CTR of zero, so I would not read anything into those cells.

### Signal check 2: March momentum

`momentum_mar` = (late-March impressions per day + 1) / (early-March impressions per day + 1). Below 1 means the page was already losing impressions inside March. Buckets are fixed, readable cut points rather than quantiles.

In [4]:
MOM_EDGES  = [0, 0.5, 0.8, 0.95, 1.05, 1.25, 2, np.inf]
MOM_LABELS = ["a_below_0.5", "b_0.5_0.8", "c_0.8_0.95", "d_flat_0.95_1.05", "e_1.05_1.25", "f_1.25_2", "g_above_2"]
df["momentum_bucket"] = pd.cut(df["momentum_mar"], MOM_EDGES, labels=MOM_LABELS)
print(f"n = {len(df):,} pages\n")
bucket_table("momentum_bucket")

n = 50,299 pages



,n,decline_rate_apr,median_imp_mar,median_imp_lost_apr
momentum_bucket,,,,
a_below_0.5,3871,0.826,1899.0,851.0
b_0.5_0.8,11357,0.619,1689.0,379.0
c_0.8_0.95,7577,0.504,1675.0,246.0
d_flat_0.95_1.05,4913,0.444,1880.0,177.0
e_1.05_1.25,7884,0.395,1943.0,131.0
f_1.25_2,10788,0.327,2092.0,18.0
g_above_2,3909,0.252,3325.0,0.0


**Verdict: CONFIRMED.**

The idea predicted that pages already sliding in late March keep sliding in April. The decline rate goes down in every step: about 0.83 for pages that halved their daily impressions inside March (n = 3,871), 0.62 for 0.5-0.8 (n = 11,357), 0.44 for flat pages (n = 4,913), and 0.25 for pages that more than doubled (n = 3,909). Every bucket has thousands of pages, so this is not a small-n effect. One honest caveat: part of this is mechanical. The label compares April to the *whole* March average, so a page that dropped in late March and then just stays at that lower level already counts as "declining". The signal is real and knowable on 31 March, but it is partly "the decline already started", not a deep insight.

### What the rule leans on, and its reason codes

Both signals came out CONFIRMED, so the rule uses both, with momentum as the main driver (stronger, cleaner pattern) and CTR-vs-position as a small boost.

- **Score** = `imp_drop_per_day` × `ctr_boost`, where `imp_drop_per_day` = max(0, early-March impressions per day − late-March impressions per day), and `ctr_boost` = 1.5 if the page had clicks and its CTR is below its tier median, else 1. Using the *absolute* drop (not the ratio) means big pages that lost a lot of impressions rank above small pages that lost a lot in percent: an editor's hour is worth more there. The 1.5 is hand-picked, not fitted.
- **Reason code (exactly one per row, first match wins):**
  - `losing_momentum_low_ctr`: momentum < 0.8 and CTR below its tier median (with clicks)
  - `losing_momentum`: momentum < 0.8
  - `mild_slip`: momentum 0.8 to 0.95
  - `holding_steady`: momentum ≥ 0.95
- **Action label:**
  - `refresh`: momentum < 0.8 and position ≤ 10 (visible on page 1 and losing)
  - `expand`: momentum < 0.8 and position 10-20 (striking distance and losing, so the page likely needs more depth to climb back)
  - `protect`: momentum ≥ 0.95 and position ≤ 10 (visible and holding, so leave it alone)
  - `monitor`: everything else (mild slips, and losing pages ranked deeper than 20 where one month says little)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The code below computes the score, reason code and action from March columns only, ranks pages inside each client (`rank_in_client`, 1 = review first) and overall (`rank_overall`), and writes the queue. Ties are broken by March impressions, then by page id, so the order is fully deterministic. The CSV holds no April columns: it is the queue an editor would get on 31 March.

In [5]:
RULE_INPUTS = ["imp_early_per_day", "imp_late_per_day", "momentum_mar", "ctr_mar", "clk_mar",
               "avg_position_mar", "imp_mar"]          # March only, all knowable on 31 March

def baseline_rule(d):
    out = pd.DataFrame(index=d.index)
    tier = pd.cut(d["avg_position_mar"], TIER_EDGES, labels=TIER_LABELS)
    ctr_vs_tier = d["ctr_mar"] / tier.map(tier_median_ctr).astype(float)
    low_ctr  = (d["clk_mar"] > 0) & (ctr_vs_tier < 1)
    losing   = d["momentum_mar"] < 0.8
    holding  = d["momentum_mar"] >= 0.95
    pos      = d["avg_position_mar"]

    out["imp_drop_per_day"] = (d["imp_early_per_day"] - d["imp_late_per_day"]).clip(lower=0)
    out["score"] = (out["imp_drop_per_day"] * np.where(low_ctr, 1.5, 1.0)).round(2)
    out["reason_code"] = np.select(
        [losing & low_ctr, losing, ~holding], ["losing_momentum_low_ctr", "losing_momentum", "mild_slip"],
        "holding_steady")
    out["action"] = np.select(
        [losing & (pos <= 10), losing & (pos > 10) & (pos <= 20), holding & (pos <= 10)],
        ["refresh", "expand", "protect"], "monitor")
    return out

df = df.join(baseline_rule(df[RULE_INPUTS]))

df = df.sort_values(["client_hash_id", "score", "imp_mar", "content_hash_id"],
                    ascending=[True, False, False, True]).reset_index(drop=True)
df["rank_in_client"] = df.groupby("client_hash_id").cumcount() + 1
overall_order = df.sort_values(["score", "imp_mar", "content_hash_id"], ascending=[False, False, True]).index
df.loc[overall_order, "rank_overall"] = np.arange(1, len(df) + 1)
df["rank_overall"] = df["rank_overall"].astype(int)

QUEUE_COLS = ["client_hash_id", "content_hash_id", "rank_in_client", "rank_overall", "score", "reason_code",
              "action", "imp_mar", "imp_drop_per_day", "momentum_mar", "avg_position_mar", "position_tier",
              "ctr_mar", "ctr_vs_tier"]
os.makedirs("work/outputs", exist_ok=True)
df[QUEUE_COLS].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv: {len(df):,} rows, {df['client_hash_id'].nunique()} clients")
print("\nRows per action:", df["action"].value_counts().to_dict())
print("Rows per reason code:", df["reason_code"].value_counts().to_dict())

Wrote work/outputs/baseline_action_score.csv: 50,299 rows, 32 clients

Rows per action: {'monitor': 21385, 'protect': 17817, 'refresh': 8298, 'expand': 2799}
Rows per reason code: {'holding_steady': 27498, 'losing_momentum': 10137, 'mild_slip': 7583, 'losing_momentum_low_ctr': 5081}


### Is the queue any good? precision@20 per client

For each client with at least 20 pages, I take its top 20 by score and measure (a) the share that actually declined in April, (b) the client's base rate, (c) the same for a random order (average of 200 seeded shuffles, seed 42), and (d) the share of the client's April lost impressions that sit in those 20 pages. Then I average over clients (macro), so one huge client does not dominate. "Biggest pages first" is shown as a reference, not as a second rule.

In [6]:
K, SEED, N_SHUFFLES = 20, 42, 200
rng = np.random.default_rng(SEED)

def top_k_stats(order_df, k=K):
    top = order_df.head(k)
    lost_total = order_df["imp_lost_apr"].sum()
    return top["is_declining"].mean(), (top["imp_lost_apr"].sum() / lost_total if lost_total > 0 else np.nan)

rows = []
for client, g in df.groupby("client_hash_id"):
    if len(g) < K:
        continue
    rule_p, rule_loss = top_k_stats(g.sort_values("rank_in_client"))
    big_p, big_loss = top_k_stats(g.sort_values(["imp_mar", "content_hash_id"], ascending=[False, True]))
    rand = [top_k_stats(g.iloc[rng.permutation(len(g))]) for _ in range(N_SHUFFLES)]
    rows.append({"pages": len(g), "base_rate": g["is_declining"].mean(),
                 "rule_p@20": rule_p, "random_p@20": np.mean([r[0] for r in rand]), "biggest_first_p@20": big_p,
                 "rule_loss_captured": rule_loss, "random_loss_captured": np.nanmean([r[1] for r in rand]),
                 "biggest_first_loss_captured": big_loss})
per_client = pd.DataFrame(rows)

summary = per_client.drop(columns="pages").mean().round(3)
print(f"Clients evaluated (>= {K} pages): {len(per_client)} of {df['client_hash_id'].nunique()} "
      f"| pages in them: {per_client['pages'].sum():,}")
print(f"Clients where the rule beats their base rate at @20: {(per_client['rule_p@20'] > per_client['base_rate']).sum()} of {len(per_client)}\n")
summary.to_frame("macro_mean_over_clients")

Clients evaluated (>= 20 pages): 22 of 32 | pages in them: 50,234
Clients where the rule beats their base rate at @20: 21 of 22



,macro_mean_over_clients
base_rate,0.489
rule_p@20,0.711
random_p@20,0.489
biggest_first_p@20,0.427
rule_loss_captured,0.241
random_loss_captured,0.102
biggest_first_loss_captured,0.289


In [7]:
# Decline rate per action label, over all pages (a sanity check that the labels mean something).
print(df.groupby("action").agg(n=("is_declining", "size"), decline_rate_apr=("is_declining", "mean"),
                               median_score=("score", "median")).round(3))

metrics = {
    "assignment": "ML-07 baseline action score",
    "decision_moment": "2026-03-31",
    "feature_window": "2026-03-01 to 2026-03-31 (month=2026-03 partition)",
    "label_window": "2026-04-01 to 2026-04-30 (month=2026-04 partition), evaluation only",
    "label_definition": "is_declining = 1 if April avg daily GSC impressions < 0.8 x March avg daily impressions",
    "slice": "gsc_data_available IS TRUE on all 31 March and 30 April days, March impressions >= 100",
    "rule": "score = max(0, early-March imp/day - late-March imp/day) x (1.5 if clicks > 0 and CTR below position-tier median else 1)",
    "n_pages": int(len(df)), "n_clients": int(df["client_hash_id"].nunique()),
    "k": K, "n_clients_evaluated": int(len(per_client)), "n_pages_in_evaluated_clients": int(per_client["pages"].sum()),
    "base_rate_pooled": round(float(df["is_declining"].mean()), 4),
    "base_rate_macro": round(float(per_client["base_rate"].mean()), 4),
    "precision_at_20_macro": round(float(per_client["rule_p@20"].mean()), 4),
    "random_precision_at_20_macro": round(float(per_client["random_p@20"].mean()), 4),
    "biggest_first_precision_at_20_macro": round(float(per_client["biggest_first_p@20"].mean()), 4),
    "loss_captured_top20_macro": round(float(per_client["rule_loss_captured"].mean()), 4),
    "random_loss_captured_top20_macro": round(float(per_client["random_loss_captured"].mean()), 4),
    "biggest_first_loss_captured_top20_macro": round(float(per_client["biggest_first_loss_captured"].mean()), 4),
    "clients_rule_beats_base_rate": int((per_client["rule_p@20"] > per_client["base_rate"]).sum()),
    "random_seed": SEED, "random_shuffles": N_SHUFFLES,
}
with open("work/outputs/baseline_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print("\nWrote work/outputs/baseline_metrics.json")

             n  decline_rate_apr  median_score
action                                        
expand    2799             0.640        19.240
monitor  21385             0.464         2.270
protect  17817             0.363         0.000
refresh   8298             0.681        34.215

Wrote work/outputs/baseline_metrics.json


**What this measures.** Over the 22 clients with at least 20 pages, the rule's top 20 had a macro precision@20 of about **0.71**, against a macro base rate of about **0.49** (random order gives the same, as expected). Its top 20 held about **24%** of each client's April lost impressions on average, against about 10% for random order. "Biggest pages first" catches a bit more of the loss (about 29%) but its precision (about 0.43) is *below* the base rate, which is the same trade-off I saw in ML-03.

**This is the bar.** The Week-5 model has to beat **precision@20 ≈ 0.71 and loss captured ≈ 24%** on the same slice, the same label and the same per-client K, ideally on held-out clients. I am freezing this rule now so the goalposts do not move. Two cautions: these numbers are in-sample for a hand rule (nothing was fitted, but I did pick the rule after seeing the bucket tables), and a good part of the momentum signal is the "decline already started" effect described in section 1.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The card asks for a hand review of the **top ten**. I take the ten highest scores **overall** (`rank_overall` 1-10), because those are the pages where the rule claims the most impressions per day are at risk. The table shows only safe columns (no ids). The April columns are shown here *only* to grade the picks after the fact; the rule never saw them.

In [8]:
top10 = df.sort_values("rank_overall").head(10).copy()
client_letter = {c: chr(65 + i) for i, c in enumerate(pd.unique(top10["client_hash_id"]))}  # short labels, not ids
top10["client"] = top10["client_hash_id"].map(client_letter)
top10["client_pages"] = top10["client_hash_id"].map(df.groupby("client_hash_id").size())
show = top10[["rank_overall", "client", "client_pages", "rank_in_client", "action", "reason_code", "score",
              "imp_mar", "imp_drop_per_day", "momentum_mar", "avg_position_mar", "ctr_vs_tier",
              "imp_apr", "is_declining"]]
print(f"Top 10 come from {top10['client'].nunique()} clients")
print(show.round({"score": 0, "imp_drop_per_day": 0, "momentum_mar": 2, "avg_position_mar": 1, "ctr_vs_tier": 2}).to_string(index=False))

Top 10 come from 4 clients
 rank_overall client  client_pages  rank_in_client  action             reason_code  score  imp_mar  imp_drop_per_day  momentum_mar  avg_position_mar  ctr_vs_tier  imp_apr  is_declining
            1      A          9721               1 refresh losing_momentum_low_ctr 4715.0  58278.0            3144.0          0.08               8.3         0.04   3066.0             1
            2      A          9721               2 refresh losing_momentum_low_ctr 4668.0  65330.0            3112.0          0.14               4.6         0.24   3648.0             1
            3      B          9191               1 refresh losing_momentum_low_ctr 4563.0  64935.0            3042.0          0.15               7.6         0.13  23240.0             1
            4      A          9721               3 refresh losing_momentum_low_ctr 3967.0 132593.0            2645.0          0.52               5.9         0.29  49347.0             1
            5      C           239              

**My top-10 review** (client letters A-D are short labels for this table only; per-day numbers come from the table: early and late March impressions per day, then April per day):

1. **#1, client A, refresh.** Why: daily impressions fell from about 3,401 (early March) to 257 (late March), momentum 0.08, at position 8.3 with CTR only 0.04× its tier median, so it gets the CTR boost. April came in at about 102 per day, so it did decline. What would make it wrong: if the early-March level was a one-off burst (news, a campaign), the "loss" is just a return to normal and a refresh would waste the editor's hour.
2. **#2, client A, refresh.** Why: about 3,613 down to 501 per day (momentum 0.14), position 4.6, CTR 0.24× tier median. April about 122 per day, declined. What would make it wrong: if the page was retired, redirected or replaced on purpose, there is nothing to refresh; the right label would be monitor or no action.
3. **#3, client B, refresh.** Why: about 3,567 down to 524 per day (momentum 0.15), position 7.6, CTR 0.13× tier median. April about 775 per day: counted as declining (0.37× the March average), but that is *above* its late-March level. What would make it wrong: the page may already be recovering, and it only "declined" because an early-March spike inflated the March average; then monitor fits better than refresh.
4. **#4, client A, refresh.** Why: one of the biggest pages (132,593 March impressions), about 5,557 down to 2,912 per day (momentum 0.52), position 5.9, CTR 0.29× tier median. April about 1,645 per day, declined. What would make it wrong: if another page on the same site took over its queries (cannibalisation), the fix is merging or re-targeting, not refreshing this page.
5. **#5, client C, refresh.** Why: top of a small client's queue (239 pages); about 3,104 down to 537 per day (momentum 0.17), position 6.0, CTR 0.35× tier median. April about 1,990 per day, *above* the March average, so it did **not** decline. What would make it wrong: exactly what happened. The early-March level looks like a short spike, and the page went back to its usual traffic. This is a clear miss.
6. **#6, client A, refresh.** Why: about 5,804 down to 3,344 per day (momentum 0.58), position 3.2, CTR 0.14× tier median. April about 943 per day, declined. What would make it wrong: a top-3 page with very low CTR may be losing clicks to search features above it rather than to weak content; then refreshing the text would not change much, and the editor should look at the results page first.
7. **#7, client D, refresh.** Why: about 2,672 down to 484 per day (momentum 0.18), position 9.0, CTR 0.34× tier median. April about 1,501 per day (0.93× the March average), so it did **not** decline. What would make it wrong: the same spike-then-return pattern as #5, plus position 9.0 sits on the page-1 edge, where small rank moves swing impressions a lot. A second miss.
8. **#8, client B, monitor.** Why: about 3,438 down to 1,272 per day (momentum 0.37), CTR 0.85× tier median, but average position 30.2, so the action rule says monitor. April about 1,344 per day, declined. What would make it wrong: the *action*, not the score. An average position of 30 can hide a main query on page 1 dragged down by long-tail queries; if so, this page needs a refresh, not monitoring.
9. **#9, client B, expand.** Why: the largest page in the slice (244,931 March impressions), about 9,414 down to 6,288 per day (momentum 0.67), position 15.2. Its CTR is 1.72× its tier median, so no CTR boost and reason code `losing_momentum`. April about 5,790 per day, declined. What would make it wrong: its high score comes from sheer size (a 33% slide on a huge page); if the drop is falling demand for the topic (seasonal), expanding the content will not bring the searches back.
10. **#10, client B, monitor.** Why: about 2,964 down to 927 per day (momentum 0.31), position 36.1, CTR 0.56× tier median. April about 897 per day: declined against the March average, but flat against late March. What would make it wrong: the drop may already be over and April is simply the new level, so the label calls it a hit while no action now would win anything back; monitor may be exactly right, which would make its rank 10 a wasted slot.

**Tally:** 8 of the 10 declined in April (by the label). The two misses (#5, #7) share one pattern, a sharp early-March level that fell and then came back.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Two checks in code: (1) the leakage check: the score is rebuilt from March-only columns and must match exactly, and no April or snapshot column is a rule input; (2) a look at the wider top 20 per client for signs of weak picks (zero-score ties, tiny clients, pages whose drop is a short spike falling back).

In [9]:
# (1) Leakage check: rebuild the score from ONLY the March rule inputs and compare.
FORBIDDEN = {"imp_apr", "is_declining", "imp_lost_apr", "content_updated_date", "last_optimized_date"}
assert not FORBIDDEN & set(RULE_INPUTS), "a future or label column is a rule input"
assert all(not c.startswith("ga4_") for c in RULE_INPUTS)
rebuilt = baseline_rule(df[RULE_INPUTS])
print("Rule inputs:", RULE_INPUTS)
print("Score rebuilt from March-only inputs matches:", bool((rebuilt["score"] == df["score"]).all()))
print("April / label / snapshot columns in the written CSV:", sorted(FORBIDDEN & set(QUEUE_COLS)) or "none")

# (2) Weak-pick checks on each client's top 20.
top20 = df[df["rank_in_client"] <= K]
sizes = df.groupby("client_hash_id").size()
print(f"\nClients with fewer than {K} pages (whole site fits in the queue): {(sizes < K).sum()} "
      f"(pages: {sizes[sizes < K].sum()})")
print(f"Top-20 slots with score 0 (pure tie, order set by March impressions): {(top20['score'] == 0).sum()} of {len(top20)}")
print(f"Top-20 slots with action 'monitor' (losing but ranked deeper than 20, or only a mild slip): {(top20['action'] == 'monitor').sum()}")
print(f"Top-10 overall share held by the single most common client: {top10['client'].value_counts().iloc[0]} of 10")
print("\nTop-20 precision by action label:")
print(top20.groupby("action").agg(n=("is_declining", "size"), decline_rate_apr=("is_declining", "mean")).round(3))

Rule inputs: ['imp_early_per_day', 'imp_late_per_day', 'momentum_mar', 'ctr_mar', 'clk_mar', 'avg_position_mar', 'imp_mar']
Score rebuilt from March-only inputs matches: True
April / label / snapshot columns in the written CSV: none

Clients with fewer than 20 pages (whole site fits in the queue): 10 (pages: 65)
Top-20 slots with score 0 (pure tie, order set by March impressions): 47 of 505
Top-20 slots with action 'monitor' (losing but ranked deeper than 20, or only a mild slip): 132
Top-10 overall share held by the single most common client: 4 of 10

Top-20 precision by action label:
           n  decline_rate_apr
action                        
expand    62             0.677
monitor  132             0.530
protect   31             0.258
refresh  280             0.743


**Weak picks I found:**

- **Spike-then-return pages (#5, #7, and partly #3).** The rule cannot tell a real slide from a short early-March surge that ends. All three had momentum below 0.2, and two of them were back at their normal level in April. *Change:* compare late March to a longer earlier window (February is also before the decision moment, so it is allowed), and only score a drop if late March is also below that longer baseline.
- **Top scores with a "monitor" action (#8, #10).** The score says "look here first" and the action says "just watch". That mismatch comes from using the impression-weighted average position, which blends many queries. *Change:* for big drops ranked 20-50, use expand or a separate "investigate" label, or use the page's main-query position instead of the average.
- **A few big sites dominate the overall list.** Clients A and B hold 8 of the overall top 10. That is why the real output is the per-client queue (`rank_in_client`), and the overall ranking is only a view.
- **Tiny clients.** 10 of 32 clients have fewer than 20 pages (65 pages in total), so their "top 20" is the whole site. All 47 zero-score slots in the top-20 lists sit in these tiny clients, ordered only by March impressions, and that is also where 30 of the 31 "protect" rows in the top 20 sit. They are not real picks, and precision@20 leaves these clients out.
- **The label rewards persistence.** #3 and #10 count as "declining" mostly because April stayed at the lower late-March level. The momentum signal partly re-measures a decline that had already started. That is fine for a baseline, but it is the easy part, and the Week-5 model must beat this bar honestly rather than through the same effect.

**Leakage check (from the cell above):** the score rebuilds exactly from the seven March-only inputs; none of `imp_apr`, `is_declining`, `imp_lost_apr`, `content_updated_date`, `last_optimized_date` or any GA4 column is an input, and the written CSV holds no April column. No product flags are used. The action labels also behave sensibly in the top 20 (decline rate: refresh 0.74, expand 0.68, monitor 0.53, protect 0.26), which is observed and in-sample, so I read it as directional only.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (pseudonymized IDs only, never printed in the review, and never used as features)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.